# Phase 3.5: Advanced Probing Controls

This notebook downloads your generated hidden states and dataset from Hugging Face, fetches the latest `advanced_probing.py` script from GitHub, and runs the final control experiments (Hop-1 Only, Shuffled-Label, Cross-Hop Generalization, and Bootstrap Error Bars).

In [ ]:
!pip install -q huggingface_hub scikit-learn
import huggingface_hub
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    huggingface_hub.login(token=hf_token)
    print("Logged in via Kaggle Secrets!")
except Exception as e:
    print("Kaggle Secrets not found. Using fallback...")
    hf_token = "REPLACE_ME_WITH_YOUR_RAW_HF_TOKEN"
    huggingface_hub.login(token=hf_token)

In [ ]:
from huggingface_hub import snapshot_download

print("Downloading dataset and hidden states from Hugging Face... (This may take a minute)")
# This will download the entire dataset repo into the current Kaggle working directory
snapshot_download(
    repo_id="AnishRacherla/LinguaFranca-Phase3", 
    repo_type="dataset", 
    local_dir="."
)
print("Download complete!")

In [ ]:
print("Pulling latest advanced_probing.py from GitHub...")
!wget -q https://raw.githubusercontent.com/ANLPproject/LinguaFranca/main/advanced_probing.py -O advanced_probing.py

print("\n--- RUNNING ADVANCED PROBING ---\n")
!python advanced_probing.py data/2wikimultihopqa/augmented.jsonl data/hidden_states

In [ ]:
import os
from huggingface_hub import HfApi

api = HfApi()
repo_id = "AnishRacherla/LinguaFranca-Phase3"

print("Uploading trained probes to Hugging Face...")
for li in range(0, 28):
    file_path = f"data/hop1_probe_layer{li}.joblib"
    if os.path.exists(file_path):
        try:
            api.upload_file(
                path_or_fileobj=file_path,
                path_in_repo=f"probes/hop1_probe_layer{li}.joblib",
                repo_id=repo_id,
                repo_type="dataset"
            )
            print(f"Uploaded Layer {li} Probe!")
        except Exception as e:
            print(f"Failed to upload Layer {li} Probe: {e}")
        
print("All probes successfully saved to Hugging Face!")